In [ ]:
import os
os.environ['HUGGINGFACEHUB_API_TOKEN'] = ""
os.environ["TAVILY_API_KEY"] = ""
os.environ["GROQ_API_KEY"] = ""

# Create **Tools**

In [7]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup
from tavily import TavilyClient
import os
from rich import print
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@tool
def web_search(query: str):
    """Search the web using Tavily and return the top 3 search results."""

    results = tavily.search(query=query, max_results=2)

    output = []

    for i, item in enumerate(results["results"], start=1):
        output.append(
            f"""Result {i}
Title: {item['title']}
URL: {item['url']}
Content: {item['content']}
"""
        )

    return "\n" + "=" * 70 + "\n".join(output)

response = web_search.invoke("What is fifa world cup?")
print(response)

======================================================================Result 1
Title: FIFA World Cup - Wikipedia
URL: https://en.wikipedia.org/wiki/FIFA_World_Cup
Content: # FIFA World Cup. This article is about the men's association football tournament. For the ongoing 2026 
event, see 2026 FIFA World Cup. For similar tournaments, see association football world cups. The **FIFA World 
Cup** is an international association football competition among the senior men's national teams of the members of 
the Fédération Internationale de Football Association (FIFA), the sport's global governing body. The tournament has
been held every four years since the inaugural tournament in 1930, with the exception of 1942 and 1946 due to World
War II. The reigning champions are Argentina, who won their third title at the 2022 World Cup by defeating France. 
In the tournament phase, 48 teams (as of the 2026 World Cup) compete for the title at venues within the host 
nation(s) over the course of about a month. As of the 2026 World Cup, 23 final tournaments have been held since the
event's inception in 1930, and a total of 80 national teams have competed.

Result 2
Title: FIFA World Cup (@fifaworldcup) · Zürich - Instagram
URL: https://www.instagram.com/fifaworldcup?hl=en
Content: ## fifaworldcup. 🏆 The official #FIFAWorldCup account on Instagram. Photo by FIFA World Cup on July 04, 
2026. May be an image of soccer, football, poster, screen and text that says 'FIFA WORLD CUP 2026 TM WORLD CHAMPION
? May be an image of text. May be an image of football, soccer, poster, card and text that says 'MOST FIFA WORLD 
CUP™ GOALS 1 BRAZIL 2 246 3 GERMANY 243 4 ARGENTINA 163 FRANCE 5 150 ITALY 128 শাড়ু noлH BoKine OWTE 2 FIFA WORLDCUP
CUP 2026'. May be an image of football, soccer, poster, flag, card, stadium and text. May be an image of football, 
soccer, poster, ball, crowd, stadium and text that says '150 FIFA WORLD CUP FIFAWORLDGUP™GOALS GOALS MBAPRE KOUNDE 
5/10 10 5 LPAMECANO 4 37 1TO FIFA WORLDCUP CU 2026'. Video by FIFA World Cup on July 04, 2026. May be an image of 
football, soccer, cleats, stadium and text.

# Tool **2**

In [8]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup
import re

@tool
def scrape_url(url: str) -> str:
    """
    Scrapes content from a given URL and extracts clean text as a single string.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unnecessary elements
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        # Extract text with spaces instead of newlines
        text = soup.get_text(separator=" ")

        # Collapse multiple spaces/newlines into single spaces
        clean_text = re.sub(r"\s+", " ", text).strip()

        # Limit to first 3000 characters (adjust as needed)
        return clean_text[:3000]

    except Exception as e:
        return f"Error scraping {url}: {str(e)}"


url = "https://docs.langchain.com/oss/python/contributing/documentation#edit-documentation"
#"https://en.wikipedia.org/wiki/FIFA_World_Cup"
result = scrape_url.invoke(url)
print(result)

Contributing to documentation - Docs by LangChain Documentation Index Fetch the complete documentation index at: 
/llms.txt Use this file to discover all available pages before exploring further. Skip to main content We welcome 
contributions to LangChain documentation, including new features, integrations , and improvements to existing docs.
​ Quick start - local development To run a local preview of the documentation: git clone 
https://github.com/langchain-ai/docs.git cd docs make install make dev This starts a development server with hot 
reload at http://localhost:3000 . Edit files in src/ and see changes immediately. Using an AI coding agent? Install
LangChain Skills to improve your agent’s performance on LangChain ecosystem tasks, then click the “Copy page” 
button on the top right of this page and paste the raw content into your agent to have it set up your environment 
automatically. If you are having issues with you local preview, try running mint update to ensure you’re using the 
latest Mintlify version. Prerequisites Required: Python 3.13+ uv - Python package manager Node.js and npm Make Git 
Optional: markdownlint-cli - npm install -g markdownlint-cli Mintlify MDX VSCode extension ​ Edit documentation 
Quick edits on GitHub For typos or small changes, edit directly on GitHub without local setup: Click Edit this page
on GitHub at the bottom of any page. Fork to your personal account. Make changes in GitHub’s web editor. Create a 
pull request. Only edit files in src/ — The build/ directory is automatically generated. Edit files in src/ 
following our writing standards . Run quality checks before submitting. Create a pull request for review. All pull 
requests must link to an issue or discussion where a solution has been approved by a maintainer. See our pull 
request requirements . Create a sharable preview build (LangChain team only) When you create or update a PR, a 
preview branch/ID is automatically generated. A comment will be left on the PR with the ID. Copy the preview 
branch’s ID from the comment In the Mintlify dashboard , click Create preview deployment Enter the preview branch’s
ID and click Create deployment Select the preview and click Visit to view To redeploy with latest changes, click 
Redeploy on the dashboard. ​ Run quality checks Before submitting changes, ensure your code passes formatting and 
linting checks: # Check broken links make broken-links # Format code automatically make format # Check for linting 
issues make lint # Fix markdown issues make lint_md_fix # Run tests to ensure your changes don't break existing 
functionality make test For more details, see the available commands section in the README . ​ Documentation types 
All documentation falls under one of four categories: How-to guides Task-oriented instructions for users who know 
what they want to accomplish. Conceptual guides Explanations that provide deeper understanding and insights. 
Reference Technical descriptions of APIs and implementation details. Tutorials Les

In [9]:
%%writefile tools.py

from langchain_core.tools import tool
from tavily import TavilyClient
import requests
from bs4 import BeautifulSoup
import os

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@tool
def web_search(query: str) -> str:
    """Search the web using Tavily."""
    results = tavily.search(query=query, max_results=3)

    output = []
    for item in results["results"]:
        output.append(
            f"Title: {item['title']}\n"
            f"URL: {item['url']}\n"
            f"Content: {item['content']}"
        )

    return "\n\n".join(output)


@tool
def scrape_url(url: str) -> str:
    """Scrape a webpage and return clean text."""

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        text = soup.get_text(separator="\n")
        lines = [line.strip() for line in text.splitlines() if line.strip()]

        return "\n".join(lines[:500])

    except Exception as e:
        return f"Could not scrape {url}\nError: {e}"

Writing tools.py


In [10]:
import importlib
import tools

importlib.reload(tools)

print(dir(tools))

[
    'BeautifulSoup',
    'TavilyClient',
    '__builtins__',
    '__cached__',
    '__doc__',
    '__file__',
    '__loader__',
    '__name__',
    '__package__',
    '__spec__',
    'os',
    'requests',
    'scrape_url',
    'tavily',
    'tool',
    'web_search'
]

In [11]:
from tools import web_search, scrape_url

# **Create Agent**

In [12]:
import os
from langgraph.prebuilt import create_react_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_agent


In [13]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0
)

agent = create_react_agent(
    model=llm,
    tools=[web_search, scrape_url]
)

/tmp/ipykernel_973/1935846399.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


# 1 Search **Agent**

In [14]:
# Create the agent
research_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="""
You are an AI Research Assistant.

Your responsibilities are:
1. Search the web for recent and reliable information.
2. When necessary, scrape the provided URL to gather more detailed information.
3. Summarize the findings into a clear and accurate answer.
4. Always prefer reliable sources.
"""
)

# Reader **Agent**

In [15]:
import os
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from tools import scrape_url

# Initialize the LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0
)

# Create the Reader Agent
reader_agent = create_agent(
    model=llm,
    tools=[scrape_url],
    system_prompt="""
You are an expert Reader Agent.

Your responsibilities are:
1. Read the content of a webpage using the scrape_url tool.
2. Extract the key information from the page.
3. Summarize the important points.
4. Ignore advertisements, navigation menus, and unrelated content.
5. If the page cannot be scraped, clearly explain that the content could not be accessed.
6. Return a concise, well-structured summary.
"""
)

# Writer **chain**

In [16]:
from langchain_core.prompts import ChatPromptTemplate

report_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an Expert Research Report Writer.

Your task is to transform gathered research into a professional, well-structured, and comprehensive report.

Instructions:

1. Carefully analyze all the provided research material.
2. Do not invent or assume facts that are not present in the provided information.
3. Combine information from multiple sources into one coherent report.
4. Remove duplicate or repetitive information.
5. Write in a professional and objective tone.
6. Use clear headings and subheadings.
7. Explain technical concepts in a way that is easy to understand.
8. Highlight important facts, statistics, dates, and trends.
9. Mention uncertainties if the available information is incomplete.
10. End with a concise conclusion.

The report must follow this structure exactly:

# Research Report

## Topic
State the research topic.

## Executive Summary
Write a brief overview (150–250 words) summarizing the report.

## Introduction
- Introduce the topic.
- Explain why it is important.
- Provide background information.

## Research Gathered
Summarize all collected information from the provided sources.
Group similar information together.
Use paragraphs and bullet points where appropriate.

## Key Findings
Identify the most important findings.
Include:
- Important facts
- Statistics
- Dates
- Trends
- Significant events
- Expert opinions (if available)

## Detailed Analysis
Provide a deeper explanation of the gathered information.
Discuss:
- Causes
- Effects
- Comparisons
- Advantages
- Disadvantages
- Challenges
- Opportunities
- Future outlook

## Conclusion
Summarize the overall findings.
Provide a balanced final assessment based only on the available evidence.

## Sources
List every source used in the report.
For each source include:
- Title
- URL

Rules:
- Never fabricate information.
- Never cite sources that were not provided.
- If some information is missing, clearly state that it was unavailable.
- Produce a detailed, organized, and readable report.
"""
        ),
        (
            "human",
            """
Research Topic:
{topic}

Research Data:
{research}
"""
        ),
    ]
)

In [17]:
from langchain_core.output_parsers import StrOutputParser

report_chain = (
    report_prompt
    | llm
    | StrOutputParser()
)

# Critic **chain**

In [18]:
from langchain_core.prompts import ChatPromptTemplate

critic_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an Expert Research Report Critic.

Your responsibility is to carefully review the research report.

Evaluate the report based on the following criteria:

1. Accuracy
- Are the claims supported by the provided research?
- Does the report avoid unsupported assumptions?

2. Completeness
- Does the report fully answer the research topic?
- Are any important points missing?

3. Structure
- Is the report logically organized?
- Does it include:
  - Topic
  - Executive Summary
  - Introduction
  - Research Gathered
  - Key Findings
  - Detailed Analysis
  - Conclusion
  - Sources

4. Clarity
- Is the writing clear and easy to understand?
- Are there unnecessary repetitions?

5. Evidence
- Are important facts, statistics, and dates included?
- Are the sources properly referenced?

6. Consistency
- Are there contradictions or inconsistencies?

Provide your review in the following format:

# Report Evaluation

## Overall Score
(Give a score out of 10)

## Strengths
(List the strong aspects of the report.)

## Weaknesses
(List the issues found.)

## Missing Information
(List important missing information.)

## Suggestions for Improvement
(Give specific recommendations.)

## Final Verdict
State whether the report is:
- Excellent
- Good
- Needs Minor Revisions
- Needs Major Revisions

Do not rewrite the report.
Only review and critique it.
"""
        ),
        (
            "human",
            """
Research Topic:
{topic}

Research Report:
{report}
"""
        )
    ]
)

In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

writer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert AI Research Report Writer.

Create a professional report using ONLY the supplied research.

Structure:

# Research Report

## Topic

## Executive Summary

## Introduction

## Research Gathered

## Key Findings

## Detailed Analysis

## Conclusion

## Sources

Write in clear professional language.
Do not invent facts.
"""
        ),
        (
            "human",
            """
Topic:
{topic}

Research:
{research}
"""
        )
    ]
)

writer_chain = writer_prompt | llm | StrOutputParser()

In [19]:
from langchain_core.output_parsers import StrOutputParser

critic_chain = (
    critic_prompt
    | llm
    | StrOutputParser()
)

# LCEL-based multi-agent research **pipeline**

In [20]:
def research_pipeline(topic: str) -> dict:
    state: ResearchState = {
        "topic": topic,
        "search_result": "",
        "scraped_content": "",
        "report": "",
        "critique": "",
    }

    # Step 1: Search
    state["search_result"] = search_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": state["topic"]
                }
            ]
        }
    )["messages"][-1].content

    # Step 2: Read/Scrape
    state["scraped_content"] = reader_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": state["search_result"]
                }
            ]
        }
    )["messages"][-1].content

    # Step 3: Generate Report
    state["report"] = report_chain.invoke(
        {
            "topic": state["topic"],
            "research": state["scraped_content"]
        }
    )

    # Step 4: Critique
    state["critique"] = critic_chain.invoke(
        {
            "topic": state["topic"],
            "report": state["report"]
        }
    )

    return state

In [24]:
def run_research_pipeline(topic: str) -> dict:

    state = {}

    # ===========================
    # STEP 1 : SEARCH AGENT
    # ===========================
    print("\n" + "="*60)
    print("STEP 1 - SEARCH AGENT")
    print("="*60)

    search_result = research_agent.invoke(
        {
            "messages": [
                (
                    "user",
                    f"Find recent, reliable and detailed information about: {topic}"
                )
            ]
        }
    )

    state["search_results"] = search_result["messages"][-1].content

    print(state["search_results"])

    # ===========================
    # STEP 2 : READER AGENT
    # ===========================
    print("\n" + "="*60)
    print("STEP 2 - READER AGENT")
    print("="*60)

    reader_result = reader_agent.invoke(
        {
            "messages": [
                (
                    "user",
                    f"""
Based on these search results about "{topic}", choose the most relevant URL.

Use the scrape_url tool to scrape the webpage.

Then produce a detailed summary.

Search Results:

{state["search_results"]}
"""
                )
            ]
        }
    )

    state["scraped_content"] = reader_result["messages"][-1].content

    print(state["scraped_content"])

    # ===========================
    # STEP 3 : WRITER
    # ===========================
    print("\n" + "="*60)
    print("STEP 3 - WRITER")
    print("="*60)

    research = f"""

SEARCH RESULTS

{state['search_results']}


SCRAPED CONTENT

{state['scraped_content']}

"""

    state["report"] = writer_chain.invoke(
        {
            "topic": topic,
            "research": research
        }
    )

    print(state["report"])

    # ===========================
    # STEP 4 : CRITIC
    # ===========================
    print("\n" + "="*60)
    print("STEP 4 - CRITIC")
    print("="*60)

    state["feedback"] = critic_chain.invoke(
        {
            "topic": topic,
            "report": state["report"]
        }
    )

    print(state["feedback"])

    return state


# ===========================
# RUN
# ===========================

topic = input("Enter research topic: ")

result = run_research_pipeline(topic)

print("\n" + "="*60)
print("FINAL REPORT")
print("="*60)
print(result["report"])

print("\n" + "="*60)
print("CRITIC FEEDBACK")
print("="*60)
print(result["feedback"])

Enter research topic: how to get AI internship


============================================================

STEP 1 - SEARCH AGENT

============================================================

To get an AI internship, it's essential to have a strong resume that highlights your interests and projects you've 
worked on. Networking is also crucial, and you can attend conferences to connect with researchers and potential 
employers. Additionally, you can search for AI internship listings on websites like Prosple, which offers curated 
listings from verified employers in the US. It's also a good idea to reach out directly to researchers or companies
you're interested in working with to inquire about potential internship opportunities.

...

# Research Report

## Topic
How to Get an AI Internship

## Executive Summary
This report provides an overview of the key steps to secure an AI internship, including building a strong resume, 
networking, and utilizing online platforms to search for internship listings. By following these steps, individuals
can increase their chances of getting an AI internship and gaining valuable experience in the field.

## Introduction
The field of Artificial Intelligence (AI) is rapidly growing, and internships provide a valuable opportunity for 
individuals to gain hands-on experience and build their skills. However, securing an AI internship can be 
competitive, and it is essential to have a strategic approach to increase one's chances of success.

## Research Gathered
The research gathered for this report is based on search results and scraped content. Although the scraped content 
was inaccessible due to a 403 Client Error, the search results provided valuable information on how to get an AI 
internship.

## Key Findings
The key findings of this research are:
* Building a strong resume that highlights interests and projects worked on is essential
* Networking with professionals in the field through conferences and other events is crucial
* Utilizing online platforms like Prosple to search for AI internship listings is recommended
* Reaching out directly to researchers or companies to inquire about potential internship opportunities is a good 
idea

## Detailed Analysis
To increase one's chances of getting an AI internship, it is essential to have a strong resume that showcases 
skills and experience in AI. Networking with professionals in the field can provide valuable connections and 
insights into potential internship opportunities. Online platforms like Prosple offer curated listings from 
verified employers, making it easier to search for AI internship listings. Additionally, reaching out directly to 
researchers or companies can help individuals inquire about potential internship opportunities and demonstrate 
their interest in the field.

## Conclusion
In conclusion, securing an AI internship requires a strategic approach that includes building a strong resume, 
networking, and utilizing online platforms to search for internship listings. By following these steps, individuals
can increase their chances of getting an AI internship and gaining valuable experience in the field.

## Sources
The sources used for this report include search results and scraped content. However, due to the inaccessibility of
the scraped content, the report relies primarily on the search results provided.

============================================================

CRITIC FEEDBACK

============================================================

# Report Evaluation

## Overall Score
6.5/10

## Strengths
* The report is well-structured and easy to follow, with a clear introduction, key findings, and conclusion.
* The report provides some useful tips for securing an AI internship, such as building a strong resume, networking,
and utilizing online platforms.
* The report highlights the importance of having a strategic approach to increase one's chances of getting an AI 
internship.

## Weaknesses
* The report lacks depth and detail in its analysis, with some points feeling somewhat generic and not specifically
tailored to AI internships.
* The report relies heavily on search results and scraped content, which may not be reliable or up-to-date sources 
of information.
* The report does not provide any specific examples or case studies of individuals who have successfully secured AI
internships, which could make the report more engaging and illustrative.
* The report mentions a 403 Client Error when trying to access scraped content, which raises concerns about the 
report's methodology and data collection.

## Missing Information
* Specific examples of successful AI internship applications or case studies.
* More detailed information on how to build a strong resume for an AI internship, such as specific skills or 
experiences that are highly valued.
* Information on how to prepare for AI internship interviews, such as common questions or topics that are likely to
be covered.
* Discussion of the importance of having a strong online presence, such as a personal website or GitHub profile, in
securing an AI internship.

## Suggestions for Improvement
* Conduct more in-depth research using a variety of sources, including academic papers, industry reports, and 
expert interviews.
* Provide more specific and detailed examples of how to secure an AI internship, such as tips for networking or 
building a strong resume.
* Consider including more visual elements, such as diagrams or infographics, to illustrate key points and make the 
report more engaging.
* Address the issue of the 403 Client Error and ensure that all sources used are reliable and accessible.

## Final Verdict
Needs Minor Revisions. The report provides a good foundation for understanding how to secure an AI internship, but 
could benefit from more depth, detail, and specific examples to make it more comprehensive and engaging. With some 
revisions to address the weaknesses and missing information, the report could be even more effective in providing 
valuable insights and advice to readers.